In [1]:
import sys
sys.path.append("/scratch/mzaffar/olaf/VPR-methods-evaluation/")
import torch
import numpy as np
import os
import datasets_ws
import faiss
from os.path import join
from glob import glob
from tqdm import tqdm
from parser import parse_arguments
from sklearn.decomposition import PCA
from vpr_models import get_model
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Subset
import torchvision.transforms as transforms

from utils import *

own database ws


In [2]:
model_folder="/scratch/mzaffar/olaf/weights/finetuned/"
base_models_folder="/scratch/mzaffar/olaf/weights/"
models_pths=sorted(glob(join(model_folder, "**", "*.pth"), recursive=True))
args=parse_arguments(["--method_pths","gsv_boq_finetuned_on_nordland.pth","gsv_crica_finetuned_on_nordland.pth","gsv_boq",
                      "--method_folder","weights/finetuned",
                      "--datasets_folder","../datasets_vg/datasets/",
                      "--dataset_name","svox/images",
                      "--batch_size","64",
                      "--gpu_id","6"
                    ])
os.environ["CUDA_VISIBLE_DEVICES"]=args.gpu_id
models_pths=[join(args.method_folder,method) for method in args.method_pths]

In [3]:
for pth in models_pths:
    #determining which model to initialize from path name
    name=pth.split(".")[0].split("/")[-1]
    model_type=name.split("_")[1]
    trainings_dataset_name=name.split("_")[0]
    finetune_dataset_name=name.split("_")[-1]
    print(name)

gsv_boq_finetuned_on_nordland
gsv_crica_finetuned_on_nordland
gsv_boq


In [4]:
class Method():
    def __init__(self,args,model_pth):
        pth_small=pth.split(".")[0].split("/")[-1]
        self.name=pth_small.split("_")[1]
        self.trainings_dataset_name=pth_small.split("_")[0]
        self.finetune_dataset_name=pth_small.split("_")[-1]
        if self.name==self.finetune_dataset_name:
            model_pth=join(base_models_folder,pth_small+".pth")
            print(f"Loading method {self.name} not finetuned")
        else:
            print(f"Loading method {self.name} finetuned on {self.finetune_dataset_name}")
            
        self.model=get_model(args,self.name,model_pth)
        self.resize=args.image_size
        self.sue_values=None
        self.predictions=None
        self.distances=None
        self.corrects_per_query=None
        self.recalls=None
        
    def evaluate(self,args,eval_ds):
        eval_ds.resize=self.resize
        model,rerank=self.model
        if args.pca==True:
            pca=self.pca
        else:
            pca=None
        model = model.eval()
        with torch.no_grad():
            print("Extracting database features for evaluation/testing")
            # For database use "hard_resize", although it usually has no effect because database images have same resolution
            eval_ds.test_method = "hard_resize"
            database_subset_ds = Subset(eval_ds, list(range(eval_ds.database_num)))
            database_dataloader = DataLoader(dataset=database_subset_ds, num_workers=args.num_workers,
                                            batch_size=args.batch_size, pin_memory=(args.device=="cuda"))

            all_features = np.empty((len(eval_ds), args.features_dim), dtype="float32")
            if rerank!=None:
                W, H, C = args.dense_feature_map_size
                all_local_features = np.empty((len(eval_ds), W, H, C), dtype="float32")

            for inputs, indices in tqdm(database_dataloader, ncols=100):
                
                if rerank!=None:
                    local_features, features = model(inputs.to(args.device))
                elif self.name=='boq':
                    features,_=model(inputs.to(args.device))
                else:
                    features = model(inputs.to(args.device))
                    
                features = features.cpu().numpy()
                
                if pca != None:
                    features = pca.transform(features)
                all_features[indices.numpy(), :] = features
                if rerank!=None:
                    local_features = local_features.cpu().numpy()
                    all_local_features[indices.numpy(), :] = local_features

            print("Extracting queries features for evaluation/testing")
            eval_ds.test_method = "hard_resize"
            test_method=eval_ds.test_method
            queries_subset_ds = Subset(eval_ds, list(range(eval_ds.database_num, eval_ds.database_num+eval_ds.queries_num)))
            queries_dataloader = DataLoader(dataset=queries_subset_ds, num_workers=args.num_workers,
                                            batch_size=args.batch_size, pin_memory=(args.device=="cuda"))
            
            for inputs, indices in tqdm(queries_dataloader, ncols=100):
                
                if test_method == "five_crops" or test_method == "nearest_crop" or test_method == 'maj_voting':
                    inputs = torch.cat(tuple(inputs))  # shape = 5*bs x 3 x 480 x 480
                
                if rerank!=None:
                    local_features, features = model(inputs.to(args.device))
                elif self.name=='boq':
                    features,_=model(inputs.to(args.device))
                else:
                    features = model(inputs.to(args.device))
                
                if test_method == "five_crops":  # Compute mean along the 5 crops
                    features = torch.stack(torch.split(features, 5)).mean(1)
                
                features = features.cpu().numpy()
                
                if pca != None:
                    features = pca.transform(features)
                all_features[indices.numpy(), :] = features
                
                if rerank!=None:
                    local_features = local_features.cpu().numpy()
                    all_local_features[indices.numpy(), :] = local_features
                
                if test_method == "nearest_crop" or test_method == 'maj_voting':  # store the features of all 5 crops
                    start_idx = eval_ds.database_num + (indices[0] - eval_ds.database_num) * 5
                    end_idx   = start_idx + indices.shape[0] * 5
                    indices = np.arange(start_idx, end_idx)
                    all_features[indices, :] = features
                else:
                    all_features[indices.numpy(), :] = features
                    if rerank!=None:
                        all_local_features[indices.numpy(), :] = local_features

        queries_features = all_features[eval_ds.database_num:]
        database_features = all_features[:eval_ds.database_num]
        if rerank!=None:
            queries_local_features = all_local_features[eval_ds.database_num:]
            database_local_features = all_local_features[:eval_ds.database_num]

        faiss_index = faiss.IndexFlatL2(args.features_dim)
        faiss_index.add(database_features)
        del database_features, all_features

        print("Calculating recalls")
        distances, predictions = faiss_index.search(queries_features, args.rerank_num)

        #### For each query, check if the predictions are correct
        positives_per_query = eval_ds.get_positives()
        # args.recall_values by default is [1, 5, 10, 20]
        recalls = np.zeros(len(args.recall_values))
        for query_index, pred in enumerate(predictions):
            for i, n in enumerate(args.recall_values):
                if np.any(np.in1d(pred[:n], positives_per_query[query_index])):
                    recalls[i:] += 1
                    break
        # Divide by the number of queries*100, so the recalls are in percentages
        recalls = recalls / eval_ds.queries_num * 100
        recalls_str =", ".join([f"R@{val}: {rec:.1f}" for val, rec in zip(args.recall_values, recalls)])

        if rerank!=None:
            print(f"First ranking recalls: {recalls_str}")
            predictions = rerank(predictions,queries_local_features,database_local_features)

            #### For each query, check if the predictions are correct
            positives_per_query = eval_ds.get_positives()

            recalls = np.zeros(len(args.recall_values))
            for query_index, pred in enumerate(predictions):
                for i, n in enumerate(args.recall_values):
                    if np.any(np.in1d(pred[:n], positives_per_query[query_index])):
                        recalls[i:] += 1
                        break
            # Divide by the number of queries*100, so the recalls are in percentages
            recalls = recalls / eval_ds.queries_num * 100
            recalls_str = ", ".join([f"R@{val}: {rec:.1f}" for val, rec in zip(args.recall_values, recalls)])
            
        self.distances=distances
        self.predictions=predictions
        self.recalls=recalls
        self.corrects_per_query=positives_per_query
        self.ordered_distances=distances[np.arange(0,4)[:,None],np.argsort(self.predictions)]
        del model, rerank, self.model
        print(f"resulting recalls of method {self.name}: {recalls_str}")
        return distances, predictions
   
boq=Method(args,models_pths[1])
val_ds=datasets_ws.BaseDataset_normal(args,datasets_folder="../datasets_vg/datasets",dataset_name=args.dataset_name,split='test')

Loading method boq not finetuned


Using cache found in /home/osverburg/.cache/torch/hub/amaralibey_bag-of-queries_main
Using cache found in /home/osverburg/.cache/torch/hub/facebookresearch_dinov2_main
/home/osverburg/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/osverburg/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/osverburg/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [ ]:
dists,preds=boq.evaluate(args,val_ds)

Extracting database features for evaluation/testing


 16%|█████████▋                                                    | 42/269 [01:26<11:09,  2.95s/it]

In [ ]:
print(boq.distances,boq.predictions)